# 🧹 01_Bis – Nettoyage et préparation des données

---

## 🎯 Objectifs
- Comprendre ce que sont les **valeurs manquantes** et pourquoi elles apparaissent
- Détecter les valeurs manquantes avec `isna()`
- Choisir entre **supprimer** (`dropna()`) ou **remplacer** (`fillna()`)
- **Renommer** des colonnes avec `rename()`
- **Convertir** le type d'une colonne avec `astype()`
- **Supprimer les doublons** avec `drop_duplicates()`

> ℹ️ **Astuce** : Les corrections sont cachées dans des blocs pliables.  
> Cliquez sur **💡 Correction** pour dérouler la solution.

In [ ]:
import pandas as pd
print("Pandas version :", pd.__version__)

---
## 📝 Partie 1 – Les valeurs manquantes : pourquoi et comment les détecter

### 🔎 Qu'est-ce qu'une valeur manquante ?

Une **valeur manquante**, c'est une case vide dans votre tableau — une information qu'on n'a pas.

**Dans la vraie vie, ça arrive tout le temps :**
- Un formulaire que l'utilisateur n'a pas rempli complètement
- Un capteur qui n'a pas transmis de mesure
- Une fusion de deux bases de données incomplètes
- Un champ optionnel laissé vide dans un CRM

**En Pandas, une valeur manquante s'appelle `NaN`** (*Not a Number*). Elle s'affiche comme `NaN` dans les colonnes numériques et comme `None` ou `NaN` dans les colonnes texte.

```
DataFrame avec valeurs manquantes :

     Nom    Âge    Service  Salaire
0  Julie   29.0         RH   3200.0
1   Marc    NaN         IT   5100.0   ← Âge manquant
2  Sonia   34.0  Marketing      NaN   ← Salaire manquant
3  Rachid  38.0        NaN   4700.0   ← Service manquant
4   Marc   45.0         IT   5100.0
```

### 🔎 Pourquoi c'est un problème ?

- Les calculs (moyenne, somme...) peuvent donner des résultats faux ou planter
- Les algorithmes de Machine Learning n'acceptent généralement pas les `NaN`
- Les graphiques peuvent être incomplets

**La première étape d'un travail sur des données réelles est toujours :** détecter et traiter les valeurs manquantes.

In [ ]:
import pandas as pd

# Tableau volontairement imparfait : valeurs manquantes + doublon
employes = pd.DataFrame({
    "Nom"    : ["Julie", "Marc",  "Sonia",     "Rachid", "Marc"],
    "Âge"    : [29,       None,    34,           38,       45],     # Marc 1 : âge manquant
    "Service": ["RH",    "IT",    "Marketing",  None,    "IT"],    # Rachid : service manquant
    "Salaire": [3200,    5100,    None,          4700,    5100]    # Sonia : salaire manquant
})
# Marc apparaît 2 fois → doublon

print("Notre tableau avec défauts :")
employes

In [ ]:
import pandas as pd

employes = pd.DataFrame({
    "Nom"    : ["Julie", "Marc",  "Sonia",     "Rachid", "Marc"],
    "Âge"    : [29,       None,    34,           38,       45],
    "Service": ["RH",    "IT",    "Marketing",  None,    "IT"],
    "Salaire": [3200,    5100,    None,          4700,    5100]
})

# isna() : carte des valeurs manquantes
# True = valeur manquante, False = valeur présente
print("=== Carte des valeurs manquantes (True = manquant) ===")
print(employes.isna())

print()

# .sum() compte les True → nombre de valeurs manquantes par colonne
print("=== Nombre de valeurs manquantes par colonne ===")
print(employes.isna().sum())

print()

# Pourcentage de valeurs manquantes par colonne
print("=== Pourcentage de valeurs manquantes ===")
print((employes.isna().sum() / len(employes) * 100).round(1).astype(str) + "%")


---
## 📝 Partie 2 – Traiter les valeurs manquantes

### 🔎 Deux stratégies : supprimer ou remplacer ?

Face à des valeurs manquantes, on a deux options principales :

| Stratégie | Méthode | Quand l'utiliser |
|-----------|---------|------------------|
| **Supprimer** les lignes incomplètes | `dropna()` | Quand on a beaucoup de données et peu de manquants — on peut se permettre de perdre quelques lignes |
| **Remplacer** les valeurs manquantes | `fillna()` | Quand chaque ligne est précieuse — on préfère une approximation plutôt que rien |

**Exemple :** 1 000 000 de lignes avec 50 valeurs manquantes → `dropna()` est raisonnable (on perd 0.005%).  
**Exemple :** 20 lignes avec 3 valeurs manquantes → `fillna()` est préférable (on ne peut pas se permettre de perdre 15% des données).

### 🔎 `dropna()` — supprimer les lignes incomplètes

```python
df.dropna()             # supprime toute ligne avec au moins 1 valeur manquante
df.dropna(subset=["Âge"])  # supprime seulement si Âge est manquant
```

### 🔎 `fillna()` — remplacer les valeurs manquantes

```python
df.fillna(0)            # remplace tout par 0
df.fillna("Inconnu")    # remplace tout par "Inconnu"

# Règles différentes par colonne :
df.fillna({
    "Âge"    : df["Âge"].mean(),      # remplacer par la moyenne
    "Service": "Non défini",           # remplacer par une valeur fixe
    "Salaire": df["Salaire"].median()  # remplacer par la médiane
})
```

> 💡 **Bonne pratique :** pour les colonnes numériques, on remplace souvent par la **moyenne** (si les données sont bien distribuées) ou la **médiane** (si des valeurs extrêmes risquent de fausser la moyenne). Pour les colonnes texte, on met une valeur comme `"Inconnu"` ou `"Non défini"`.

In [ ]:
import pandas as pd

employes = pd.DataFrame({
    "Nom"    : ["Julie", "Marc",  "Sonia",     "Rachid", "Marc"],
    "Âge"    : [29,       None,    34,           38,       45],
    "Service": ["RH",    "IT",    "Marketing",  None,    "IT"],
    "Salaire": [3200,    5100,    None,          4700,    5100]
})

# dropna() : supprimer toute ligne avec au moins 1 valeur manquante
print("=== dropna() : lignes supprimées ===")
print(employes.dropna())
print(f"→ {len(employes)} lignes avant, {len(employes.dropna())} après")

print()

# dropna(subset) : supprimer uniquement si une colonne spécifique est manquante
print("=== dropna(subset=[Salaire]) : supprimer seulement si Salaire est manquant ===")
print(employes.dropna(subset=["Salaire"]))

In [ ]:
import pandas as pd

employes = pd.DataFrame({
    "Nom"    : ["Julie", "Marc",  "Sonia",     "Rachid", "Marc"],
    "Âge"    : [29,       None,    34,           38,       45],
    "Service": ["RH",    "IT",    "Marketing",  None,    "IT"],
    "Salaire": [3200,    5100,    None,          4700,    5100]
})

moy_age     = employes["Âge"].mean()
median_sal  = employes["Salaire"].median()
print(f"Moyenne d'âge pour remplacement  : {moy_age:.1f}")
print(f"Médiane salaire pour remplacement : {median_sal:.0f} €")

# fillna() avec des règles différentes par colonne
employes_propre = employes.fillna({
    "Âge"    : moy_age,
    "Service": "Non défini",
    "Salaire": median_sal
})

print("\n=== Après fillna() ===")
print(employes_propre)

# Vérification : plus aucune valeur manquante ?
print("\nValeurs manquantes restantes :")
print(employes_propre.isna().sum())

---
## 📝 Partie 3 – Renommer des colonnes

### 🔎 `rename()` — changer le nom d'une ou plusieurs colonnes

Il arrive souvent que les noms de colonnes dans un fichier CSV soient peu lisibles, en anglais, avec des espaces ou des caractères spéciaux. `rename()` permet de les corriger.

```python
# Renommer une colonne
df.rename(columns={"ancien_nom": "nouveau_nom"})

# Renommer plusieurs colonnes en une fois
df.rename(columns={
    "salary" : "Salaire (€)",
    "dept"   : "Service",
    "age"    : "Âge"
})
```

> ⚠️ Comme `drop()`, `rename()` retourne une **copie** — pour que le renommage soit permanent, réassignez : `df = df.rename(...)`

In [ ]:
import pandas as pd

employes_propre = pd.DataFrame({
    "Nom"    : ["Julie", "Marc", "Sonia", "Rachid", "Marc"],
    "Âge"    : [29, 36.5, 34, 38, 45],
    "Service": ["RH", "IT", "Marketing", "Non défini", "IT"],
    "Salaire": [3200, 5100, 4850, 4700, 5100]
})

print("=== Avant renommage ===")
print(employes_propre.columns.tolist())

# Renommer Salaire → Salaire (€)
employes_propre = employes_propre.rename(columns={
    "Salaire": "Salaire (€)",
    "Âge"    : "Âge (ans)"
})

print("\n=== Après renommage ===")
print(employes_propre.columns.tolist())
print(employes_propre)

---
## 📝 Partie 4 – Convertir le type d'une colonne

### 🔎 `astype()` — changer le type de données

Quand on remplace des valeurs manquantes dans une colonne entière (`int64`) par la moyenne (un flottant), Pandas convertit automatiquement la colonne en `float64`. Résultat : `29.0` au lieu de `29`, `34.0` au lieu de `34`…

Pour revenir à des entiers lisibles, on utilise `astype(int)` :

```python
df["Âge"] = df["Âge"].astype(int)     # float64 → int64
df["Prix"] = df["Prix"].astype(float)  # int64   → float64
df["Actif"] = df["Actif"].astype(bool) # int (0/1) → bool (False/True)
```

> ⚠️ **Attention :** `astype(int)` va planter s'il reste des `NaN` dans la colonne. Il faut d'abord faire `fillna()` avant de convertir.

**Pourquoi c'est important ?**
- Une colonne de type `object` (texte) ne peut pas être utilisée dans des calculs numériques
- Une colonne de type `float` prend plus de mémoire qu'un `int`
- Les types incorrects peuvent provoquer des erreurs dans les graphiques ou les algorithmes

In [ ]:
import pandas as pd

employes = pd.DataFrame({
    "Nom"    : ["Julie", "Marc", "Sonia", "Rachid", "Marc"],
    "Âge"    : [29.0, 36.5, 34.0, 38.0, 45.0],   # float à cause du fillna
    "Salaire": [3200, 5100, 4850, 4700, 5100]
})

print("=== Avant conversion ===")
print(employes.dtypes)
print(employes[["Âge"]])

# Convertir Âge de float64 vers int64
employes["Âge"] = employes["Âge"].astype(int)

print("\n=== Après conversion (float → int) ===")
print(employes.dtypes)
print(employes[["Âge"]])
# → 29.0 devient 29, 36.5 devient 36 (troncature !)

---
## 📝 Partie 5 – Supprimer les doublons

### 🔎 `drop_duplicates()` — éliminer les lignes répétées

Un **doublon** est une ligne qui apparaît plusieurs fois avec les mêmes valeurs. Ça arrive souvent quand on fusionne plusieurs fichiers ou qu'on exporte plusieurs fois le même rapport.

```python
df.drop_duplicates()                    # supprime les lignes 100% identiques
df.drop_duplicates(subset=["Nom"])      # supprime si la colonne Nom est dupliquée
df.drop_duplicates(keep="last")         # garde la dernière occurrence (défaut : "first")
```

**Comment détecter les doublons avant de les supprimer ?**

```python
df.duplicated()           # True/False : cette ligne est-elle un doublon ?
df.duplicated().sum()     # combien de doublons au total ?
df[df.duplicated()]       # afficher uniquement les doublons
```

In [ ]:
import pandas as pd

employes = pd.DataFrame({
    "Nom"    : ["Julie", "Marc",  "Sonia",     "Rachid", "Marc"],
    "Âge"    : [29,       45,      34,           38,       45],
    "Service": ["RH",    "IT",    "Marketing",  "IT",    "IT"],
    "Salaire": [3200,    5100,    4850,          4700,    5100]
})

print("=== Tableau original ===")
print(employes)

# Détecter les doublons
print("\n=== Lignes dupliquées (True = doublon) ===")
print(employes.duplicated())
print(f"Nombre de doublons : {employes.duplicated().sum()}")

# Afficher les doublons
print("\n=== Les lignes dupliquées ===")
print(employes[employes.duplicated()])

# Supprimer les doublons
employes_clean = employes.drop_duplicates()
print("\n=== Après drop_duplicates() ===")
print(employes_clean)
print(f"→ {len(employes)} lignes avant, {len(employes_clean)} après")

---
## 🧩 Exercice 1 – Nettoyage complet d'un tableau RH

Vous avez reçu ce tableau d'employés depuis un export du logiciel RH. Il contient plusieurs problèmes :

```python
employes = pd.DataFrame({
    "Nom"     : ["Julie", "Marc", "Sonia", "Rachid", "Marc",  "Nina"],
    "Âge"     : [29,       None,   34,      38,        45,      None],
    "Service" : ["RH",    "IT",   None,    "IT",      "IT",    "Marketing"],
    "Salaire" : [3200,    5100,   4100,    4700,      5100,    None]
})
```

**Effectuez le nettoyage suivant dans l'ordre :**
1. Affichez le nombre de valeurs manquantes par colonne
2. Remplacez les valeurs manquantes :
   - `Âge` → moyenne des âges (arrondie)
   - `Service` → `"Non défini"`
   - `Salaire` → médiane des salaires
3. Supprimez les doublons
4. Convertissez `Âge` en entier
5. Renommez `"Salaire"` en `"Salaire (€)"`
6. Affichez le tableau final propre et sa structure avec `info()`

In [ ]:
import pandas as pd

employes = pd.DataFrame({
    "Nom"    : ["Julie", "Marc", "Sonia", "Rachid", "Marc",  "Nina"],
    "Âge"    : [29,       None,   34,      38,        45,      None],
    "Service": ["RH",    "IT",   None,    "IT",      "IT",    "Marketing"],
    "Salaire": [3200,    5100,   4100,    4700,      5100,    None]
})

# TODO 1 : valeurs manquantes par colonne
print("=== Valeurs manquantes ===")
print(...)

# TODO 2 : fillna avec règles par colonne
employes_clean = employes.fillna({
    "Âge"    : ...,
    "Service": ...,
    "Salaire": ...
})

# TODO 3 : supprimer les doublons
employes_clean = ...

# TODO 4 : convertir Âge en int
employes_clean["Âge"] = ...

# TODO 5 : renommer Salaire
employes_clean = employes_clean.rename(columns={...})

# TODO 6 : afficher le résultat
print("\n=== Tableau final ===")
print(employes_clean)
print()
employes_clean.info()

<details>
<summary>💡 Correction</summary>

```python
import pandas as pd

employes = pd.DataFrame({
    "Nom"    : ["Julie", "Marc", "Sonia", "Rachid", "Marc",  "Nina"],
    "Âge"    : [29,       None,   34,      38,        45,      None],
    "Service": ["RH",    "IT",   None,    "IT",      "IT",    "Marketing"],
    "Salaire": [3200,    5100,   4100,    4700,      5100,    None]
})

# 1
print("=== Valeurs manquantes ===")
print(employes.isna().sum())
# Âge : 2, Service : 1, Salaire : 1

# 2 — fillna avec règles différentes par colonne
employes_clean = employes.fillna({
    "Âge"    : employes["Âge"].mean(),         # moyenne → 36.5
    "Service": "Non défini",
    "Salaire": employes["Salaire"].median()     # médiane → 4600
})

# 3 — Marc apparaît 2 fois avec les mêmes valeurs → 1 doublon supprimé
employes_clean = employes_clean.drop_duplicates()

# 4 — float → int (attention : astype(int) tronque 36.5 → 36)
employes_clean["Âge"] = employes_clean["Âge"].astype(int)

# 5
employes_clean = employes_clean.rename(columns={"Salaire": "Salaire (€)"})

# 6
print("\n=== Tableau final ===")
print(employes_clean)
print()
employes_clean.info()
# → 5 lignes (Marc en double supprimé), colonnes propres, aucun NaN
```
</details>

---
## 🧩 Exercice 2 – Nettoyage d'un fichier de ventes

Vous avez reçu un export de ventes avec des données incorrectes :

```python
ventes = pd.DataFrame({
    "produit"  : ["Pizza", "Burger", "Pizza", "Sushi",  "Burger", "Tacos"],
    "quantite" : [120,      None,     150,     80,       None,      60],
    "prix_unit": [12.5,     8.9,     12.5,    15.0,     8.9,      9.5],
    "region"   : ["Nord",  "Sud",   "Nord",  "Est",    "Sud",    None]
})
```

**Questions :**
1. Combien de valeurs manquantes par colonne ?
2. Pour `quantite` : remplacez par la **médiane**. Pour `region` : remplacez par `"Inconnue"`
3. Supprimez les lignes en **double** (même produit + même prix + même région)
4. Ajoutez une colonne `"chiffre_affaires"` = quantite × prix_unit
5. Renommez `"prix_unit"` en `"Prix unitaire (€)"` et `"quantite"` en `"Quantité"`
6. Quel produit génère le plus de chiffre d'affaires ?

In [ ]:
import pandas as pd

ventes = pd.DataFrame({
    "produit"  : ["Pizza", "Burger", "Pizza", "Sushi",  "Burger", "Tacos"],
    "quantite" : [120,      None,     150,     80,       None,      60],
    "prix_unit": [12.5,     8.9,     12.5,    15.0,     8.9,      9.5],
    "region"   : ["Nord",  "Sud",   "Nord",  "Est",    "Sud",    None]
})
print("Tableau original :")
print(ventes)

# TODO 1 : valeurs manquantes
print("\nValeurs manquantes :")
print(...)

# TODO 2 : fillna
ventes_clean = ventes.fillna({...})

# TODO 3 : doublons
ventes_clean = ...
print(f"\n{len(ventes)} lignes → {len(ventes_clean)} après dédoublonnage")

# TODO 4 : colonne chiffre_affaires
ventes_clean["chiffre_affaires"] = ...

# TODO 5 : renommer
ventes_clean = ventes_clean.rename(columns={...})

print("\nTableau final :")
print(ventes_clean)

# TODO 6 : produit avec le plus de CA
idx_max = ventes_clean["chiffre_affaires"].idxmax()
print(f"\nProduit avec le plus de CA : {ventes_clean.loc[idx_max, 'produit']}")

<details>
<summary>💡 Correction</summary>

```python
import pandas as pd

ventes = pd.DataFrame({
    "produit"  : ["Pizza", "Burger", "Pizza", "Sushi",  "Burger", "Tacos"],
    "quantite" : [120,      None,     150,     80,       None,      60],
    "prix_unit": [12.5,     8.9,     12.5,    15.0,     8.9,      9.5],
    "region"   : ["Nord",  "Sud",   "Nord",  "Est",    "Sud",    None]
})

# 1
print(ventes.isna().sum())
# quantite : 2, region : 1

# 2
ventes_clean = ventes.fillna({
    "quantite": ventes["quantite"].median(),   # 105.0 (médiane de 60,80,120,150)
    "region"  : "Inconnue"
})

# 3 — Pizza/Nord et Burger/Sud apparaissent 2 fois chacun
ventes_clean = ventes_clean.drop_duplicates()
print(f"\n{len(ventes)} lignes → {len(ventes_clean)} après dédoublonnage")

# 4
ventes_clean["chiffre_affaires"] = ventes_clean["quantite"] * ventes_clean["prix_unit"]

# 5
ventes_clean = ventes_clean.rename(columns={
    "prix_unit": "Prix unitaire (€)",
    "quantite" : "Quantité"
})

print("\nTableau final :")
print(ventes_clean)

# 6
idx_max = ventes_clean["chiffre_affaires"].idxmax()
print(f"\nProduit avec le plus de CA : {ventes_clean.loc[idx_max, 'produit']}")
# → Pizza (150 × 12.5 = 1875 €)
```
</details>

---
## ✅ Récapitulatif

| Concept | Méthode | Ce qu'il faut retenir |
|---------|---------|----------------------|
| **Valeur manquante** | `NaN` | Case vide — peut fausser les calculs et planter les algorithmes |
| **Détecter** | `isna()` / `isna().sum()` | Carte ou comptage des valeurs manquantes par colonne |
| **Supprimer** | `dropna()` | Supprime les lignes avec au moins 1 NaN — radical, utiliser si on a beaucoup de données |
| **Remplacer** | `fillna({col: valeur})` | Remplace par une valeur fixe, une moyenne ou une médiane |
| **Renommer** | `rename(columns={ancien: nouveau})` | Changer le nom d'une ou plusieurs colonnes |
| **Convertir le type** | `astype(int)` / `astype(float)` | Changer le type d'une colonne — faire `fillna()` avant ! |
| **Doublons** | `duplicated()` / `drop_duplicates()` | Détecter puis supprimer les lignes identiques |

**L'ordre recommandé pour nettoyer un DataFrame :**
1. `info()` + `isna().sum()` → comprendre ce qu'on a
2. `fillna()` ou `dropna()` → traiter les valeurs manquantes
3. `drop_duplicates()` → éliminer les doublons
4. `astype()` → corriger les types
5. `rename()` → harmoniser les noms de colonnes

---
📘 **Prochain notebook → `01_Ter` : Sélection avancée avec `loc`, `iloc` et `apply()`**